# Performance Comparison

Chunking comparison, CAG simulation, CRAG impact and cost analysis.

In [1]:
from prime_land.chat_service.cag_cache import CAGCache
from prime_land.chat_service.cag_service import CAGService
from prime_land.chat_service.crag_service import CRAGService
from prime_land.chat_service.rag_service import RAGService
from prime_land.config import CONFIG, get_path, load_faqs
from prime_land.evaluation.cag_eval import simulate_cag
from prime_land.evaluation.chunking_eval import evaluate_chunking
from prime_land.evaluation.cost_analysis import analyze_cost
from prime_land.evaluation.crag_eval import evaluate_crag
from prime_land.ingest_document.indexer import collection_name, get_client, open_store
from prime_land.providers import get_embeddings, get_llm

embeddings = get_embeddings()
llm = get_llm()
client = get_client()
stores = {strategy: open_store(client, embeddings, strategy) for strategy in CONFIG["chunking"]["strategies"]}

c:\Users\Sasiri_A\Documents\projects\python\prime_land\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 540.45it/s]


## Chunking strategy comparison

In [2]:
chunking_table = evaluate_chunking(stores, embeddings, llm)
chunking_table

evaluating 1/50 [semantic]: Which gated house community is available in Kalutara?
evaluating 2/50 [semantic]: What is the price of land at Solara Meepe?
evaluating 3/50 [semantic]: Tell me about Veloria in Bokundara
evaluating 4/50 [semantic]: Which houses are available in Thalawathugoda?
evaluating 5/50 [semantic]: What amenities does Kaloora offer?
evaluating 6/50 [semantic]: Are there lands near Galle?
evaluating 7/50 [semantic]: What is the payment plan for Kaloora?
evaluating 8/50 [semantic]: Where is Scottish Island located?
evaluating 9/50 [semantic]: Which land is available in Kadawatha?
evaluating 10/50 [semantic]: How can I contact Prime Lands?
evaluating 11/50 [fixed]: Which gated house community is available in Kalutara?
evaluating 12/50 [fixed]: What is the price of land at Solara Meepe?
evaluating 13/50 [fixed]: Tell me about Veloria in Bokundara
evaluating 14/50 [fixed]: Which houses are available in Thalawathugoda?
evaluating 15/50 [fixed]: What amenities does Kaloora o

,strategy,precision_at_5,recall_at_5,answer_relevance,latency_s,score
4,sliding,0.84,1.0,0.722,0.782,2.562
2,parent_child,0.82,1.0,0.717,0.729,2.537
3,semantic,0.78,1.0,0.716,0.775,2.496
0,fixed,0.74,1.0,0.713,0.776,2.453
1,late,0.68,1.0,0.714,0.735,2.394


## CAG cache effectiveness

In [3]:
strategy = CONFIG["retrieval"]["strategy"]
rag = RAGService(stores[strategy], embeddings, llm, strategy=strategy)
cache = CAGCache(embeddings, get_path("cache_dir"), CONFIG["cag"]["similarity_threshold"], CONFIG["cag"]["history_ttl_hours"])
cag = CAGService(rag, cache)
cag.warm_faqs(load_faqs())
cag_stats = simulate_cag(cag, load_faqs())
cag_stats

simulating 1/100 hit 0.2s
simulating 2/100 hit 0.0s
simulating 3/100 hit 0.0s
simulating 4/100 hit 0.1s
simulating 5/100 hit 0.1s
simulating 6/100 hit 0.1s
simulating 7/100 miss 1.1s
simulating 8/100 hit 0.0s
simulating 9/100 hit 0.1s
simulating 10/100 hit 0.0s
simulating 11/100 hit 0.1s
simulating 12/100 hit 0.0s
simulating 13/100 hit 0.0s
simulating 14/100 hit 0.1s
simulating 15/100 hit 0.0s
simulating 16/100 hit 0.0s
simulating 17/100 hit 0.0s
simulating 18/100 hit 0.0s
simulating 19/100 hit 0.1s
simulating 20/100 hit 0.1s
simulating 21/100 hit 0.0s
simulating 22/100 hit 0.0s
simulating 23/100 hit 0.0s
simulating 24/100 hit 0.1s
simulating 25/100 miss 0.8s
simulating 26/100 hit 0.1s
simulating 27/100 hit 0.0s
simulating 28/100 hit 0.0s
simulating 29/100 miss 1.0s
simulating 30/100 hit 0.1s
simulating 31/100 hit 0.0s
simulating 32/100 hit 0.1s
simulating 33/100 hit 0.0s
simulating 34/100 miss 1.0s
simulating 35/100 hit 0.0s
simulating 36/100 hit 0.0s
simulating 37/100 hit 0.0s
simula

{'total_queries': 100,
 'hits': 94,
 'misses': 6,
 'faq_hits': 88,
 'history_hits': 6,
 'hit_rate': 0.94,
 'avg_hit_latency_s': 0.049,
 'avg_miss_latency_s': 0.961,
 'latency_improvement_pct': 94.9,
 'cost_per_uncached_query_usd': 0.000202,
 'estimated_savings_usd': 0.019035,
 'api_calls_avoided': 94}

## CRAG correction impact

In [4]:
crag = CRAGService(rag, llm)
crag_table, crag_summary = evaluate_crag(rag, crag, llm)
print(crag_summary)
crag_table

comparing 1/20: Which gated house community is available in Kalutara?
comparing 2/20: What is the price of land at Solara Meepe?
comparing 3/20: Tell me about Veloria in Bokundara
comparing 4/20: Which houses are available in Thalawathugoda?
comparing 5/20: What amenities does Kaloora offer?
comparing 6/20: Are there lands near Galle?
comparing 7/20: What is the payment plan for Kaloora?
comparing 8/20: Where is Scottish Island located?
comparing 9/20: Which land is available in Kadawatha?
comparing 10/20: How can I contact Prime Lands?
comparing 11/20: Which property is best for a family wanting security and a pool?
comparing 12/20: What is the cheapest house I can buy?
comparing 13/20: Can I pay in installments without interest?
comparing 14/20: Which projects are close to the highway?
comparing 15/20: I want a quiet land by the beach in the south
comparing 16/20: Do you offer bank loan facilities?
comparing 17/20: Which apartments are available?
comparing 18/20: Show me upcoming pro

,question,rag_confidence,crag_initial_confidence,crag_final_confidence,correction_applied,rag_quality,crag_quality,confidence_gain,quality_gain
0,Which gated house community is available in Ka...,0.900,0.900,0.900,False,5,5,0.000,0
1,What is the price of land at Solara Meepe?,0.849,0.849,0.849,False,5,5,0.000,0
2,Tell me about Veloria in Bokundara,0.919,0.919,0.919,False,5,5,0.000,0
3,Which houses are available in Thalawathugoda?,0.833,0.833,0.833,False,5,5,0.000,0
4,What amenities does Kaloora offer?,0.744,0.744,0.744,False,5,5,0.000,0
5,Are there lands near Galle?,0.839,0.839,0.839,False,5,5,0.000,0
6,What is the payment plan for Kaloora?,0.855,0.855,0.855,False,1,1,0.000,0
7,Where is Scottish Island located?,0.411,0.411,0.762,True,5,1,0.351,-4
8,Which land is available in Kadawatha?,1.000,1.000,1.000,False,5,5,0.000,0
9,How can I contact Prime Lands?,1.000,1.000,1.000,False,5,5,0.000,0


## Cost analysis

In [5]:
vector_count = client.count(collection_name(strategy)).count
analyze_cost(cag_stats, vector_count)

{'monthly_queries': 75000,
 'cache_hit_rate': 0.94,
 'api_cost_without_cache_usd': 15.19,
 'api_cost_with_cache_usd': 0.91,
 'storage_cost_usd': 0.0008,
 'monthly_system_cost_usd': 0.91,
 'monthly_cache_savings_usd': 14.28,
 'human_agent_cost_usd': 37500.0,
 'roi_percent': 4111418.6}

In [6]:
client.close()